# 06 Labels and cohorts

**What this does.** Keeps every human grade of a target-comp pair, under the rubric it was given with, and serves
them to the other components as named cohorts (`scripts/cohorts.py`). The pair ledger counts every pair a person or
a model judged, with the dates and rubric or prompt versions.

| cohort | who | rubric |
|---|---|---|
| `dev_293` (default) | Barbara | rounds 2-3 old 0/1/2 form |
| `round1_36` | Riya, Barbara, Matthew | round 1 (majority of three) |
| `matthew_v2_262` | Matthew | rubric v2: four yes/no questions, score derived |
| `matthew_v3_311` | Matthew | rubric v3: overall call + four yes/no questions, score derived |
| `barbara_v3_311` | Barbara | rubric v3, the same 311 pairs |
| `v3_agreed` | Matthew and Barbara | rubric v3: grade only where both answer all four questions alike |
| `v3_final` | Matthew and Barbara, GPT-Terra on split outcomes | rubric v3 headline: same outcome, splits tie-broken 2 of 3 |
| `final_holdout_24` | Barbara | direct-peer Yes/No; **locked** to 10_final_holdout |

A blank grade is never a 0. Barbara's labels are one reviewer's judgment and are pending Michael's review.

**Changelog**
- 2026-11-18: created from `v1/notebooks/golden_set.ipynb`, `golden_set_showcase.ipynb`, Matthew's rubric v2
  (`matt/rubric-yes-no`) and the pair ledger. Added `cohorts.py`. All label tables rebuild exactly.
- 2026-11-27: rubric v3 filled by Matthew and Barbara (323/323). Cohorts `matthew_v3_311`, `barbara_v3_311`,
  `v3_agreed`; section 3.7; checks for the filled sheet and the pinned scores run.
- 2026-11-28: GPT-Terra judge on all 323 pairs (`judge_v3.py`), headline labels `rubric_v3.py final` (team ruling 2026-11-27); cohort `v3_final`; section 3.8; checks for the judge run and the final labels.


In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v2" / "current.toml").exists())
sys.path.insert(0, str(ROOT / "v2" / "common"))
import paths, nbtools as nb
paths.setup()
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 80)
cur = paths.current()
checks = nb.Checks()

In [2]:
import json, glob, cohorts
pd.Series(cur["labels"])

default_cohort                                                                                      dev_293
cohorts           [dev_293, round1_36, matthew_v2_262, matthew_v3_311, barbara_v3_311, v3_agreed, v3_final]
locked_cohorts                                                                           [final_holdout_24]
dtype: object

## 2. Inputs

In [3]:
L = "outputs-local/v1a/06_labels/"
nb.inputs([L + "barbara_293/sonnet_label.xlsx", L + "barbara_293/review/pairs.csv", L + "round1/reviewer_labels_20261028.json",
           L + "round2/round2_sonnet_sheet.csv", L + "round3/round3_sheet.csv", L + "round3/round3_sheet_v2.csv",
           L + "round3/round3_scores_v2.csv",
           cur["rubric_v3"]["sheet"], cur["rubric_v3"]["filled"],
           cur["rubric_v3"]["scores"] + "/scores.csv", cur["rubric_v3"]["scores"] + "/consensus.csv"])


,input,rows,sha256,status
0,outputs-local/v1a/06_labels/barbara_293/sonnet_label.xlsx,NaN,232faa865e0b,matches seed
1,outputs-local/v1a/06_labels/barbara_293/review/pairs.csv,293.0,3ea5583a922b,matches seed
2,outputs-local/v1a/06_labels/round1/reviewer_labels_20261028.json,NaN,ee07b1c14fee,matches seed
3,outputs-local/v1a/06_labels/round2/round2_sonnet_sheet.csv,31.0,6ce91938cbd6,matches seed
4,outputs-local/v1a/06_labels/round3/round3_sheet.csv,262.0,744d67635604,matches seed
5,outputs-local/v1a/06_labels/round3/round3_sheet_v2.csv,262.0,228c924cfe5d,matches seed
6,outputs-local/v1a/06_labels/round3/round3_scores_v2.csv,262.0,65b55cbcf7c8,matches seed
7,outputs-local/v2/06_labels/rubric_v3/rubric_v3_sheet.csv,338.0,7563f9800404,new in v1a
8,outputs-local/v2/06_labels/rubric_v3/filled/rubric_v3_filled_matthew_barbara_20261127.csv,338.0,d19dfe3f48b8,new in v1a
9,outputs-local/v2/06_labels/rubric_v3/scores/20261128T001954Z/scores.csv,323.0,c6d0bc5afb8d,new in v1a


## 3. Results

### 3.1 The cohorts

In [4]:
cohorts.table()

,cohort,who,rubric,pairs,targets,grade 2,grade 1,grade 0,not graded
0,dev_293,Barbara,rounds 2-3 old 0/1/2 form,293.0,45.0,24.0,178.0,91.0,0.0
1,round1_36,"Riya, Barbara, Matthew (majority of 3)","round 1: 0/1/2, blank = 0 with one of five reasons",36.0,2.0,4.0,7.0,21.0,4.0
2,matthew_v2_262,Matthew,"rubric v2: four yes/no questions, score derived",262.0,40.0,60.0,124.0,78.0,0.0
3,matthew_v3_311,Matthew,"rubric v3: overall call + four yes/no questions, score derived",311.0,67.0,102.0,69.0,140.0,0.0
4,barbara_v3_311,Barbara,"rubric v3: overall call + four yes/no questions, score derived",311.0,67.0,135.0,53.0,123.0,0.0
5,v3_agreed,Matthew and Barbara (agreed),rubric v3: grade only where both answer all four questions alike,311.0,67.0,101.0,8.0,71.0,131.0
6,v3_final,"Matthew and Barbara, LLM judge on split outcomes","rubric v3: same outcome, split outcomes tie-broken 2 of 3",311.0,67.0,106.0,78.0,127.0,0.0
7,final_holdout_24,Barbara,direct-peer Yes/No (Yes = 2; No is not a 0) [locked],NaN,NaN,NaN,NaN,NaN,NaN


### 3.2 Every pair judged, by stage (pair ledger)

In [5]:
ledger = paths.latest("06_labels", "pair_ledger")
long = pd.read_csv(ledger / "judgments.csv", dtype={"target_id": str, "comp_id": str})
j = long[long["judged"]]
print("ledger run:", ledger.name)
pd.DataFrame({stage: {"sources": g["source"].nunique(), "judgments": len(g),
                      "unique pairs": len(g[["target_id", "comp_id"]].drop_duplicates()),
                      "targets": g["target_id"].nunique()} for stage, g in j.groupby("stage")}).T

ledger run: 20261119T014543Z


,sources,judgments,unique pairs,targets
agent_loop,18,1153,365,47
human_review,7,687,353,47
llm_judge,2,2275,2275,201
llm_screen,4,12326,5698,373


### 3.3 Rubric and prompt spans

In [6]:
spans = pd.read_csv(ledger / "spans.csv")
spans[["stage", "rubric", "who", "first", "last", "judgments", "unique_pairs_judged", "targets", "sent_not_judged"]].fillna("")

,stage,rubric,who,first,last,judgments,unique_pairs_judged,targets,sent_not_judged
0,llm_screen,binary screen prompt 2026-10-25a,claude-haiku-5-5,2026-10-27,2026-10-27,4996,4996,373,
1,human_review,"round 1: 0/1/2, blank = 0 with one of five reasons",Barbara;Matthew;Riya,2026-10-28,2026-10-28,108,36,2,
2,llm_judge,"cited grades, 0/1/2 (v0 holdout)",GPT-6.1 Sol,2026-10-28,2026-10-28,1982,1982,156,
3,agent_loop,agent prompt 2026-11-05a,sonnet agent,2026-11-05,2026-11-06,505,341,47,
4,agent_loop,agent prompt 2026-11-05b,sonnet agent,2026-11-05,2026-11-05,146,71,7,
5,llm_screen,binary screen prompt 2026-11-04a,claude-haiku-5-5;claude-sonnet-5-5,2026-11-05,2026-11-19,7330,3148,373,
6,human_review,"rounds 2-3 old form: 0/1/2, six reasons, multi-business rule",Barbara,2026-11-10,2026-11-10,293,293,45,human:round2_sheet:reviewer_a (67);human:round2_sheet:reviewer_b (67);human:round2_she...
7,llm_judge,"source-cited one-shot judge, 0/1/2",GPT-6.1 Sol,2026-11-10,2026-11-10,293,293,45,
8,agent_loop,agent prompt 2026-11-05b + facet scores,sonnet agent,2026-11-12,2026-11-18,502,313,45,
9,human_review,"rubric v2: four yes/no questions, score derived",Matthew,2026-11-13,2026-11-13,262,262,40,


### 3.4 Round 1: three reviewers on the same 36 pairs

In [7]:
r1 = cohorts.load("round1_36")
s = r1[["score_riya", "score_barbara", "score_matthew"]]
print("unanimous:", int((s.nunique(axis=1) == 1).sum()), "of 36; with a majority grade:", int(r1["grade"].notna().sum()))
names = ["riya", "barbara", "matthew"]
pd.DataFrame([{"pair": f"{a} vs {b}", "exact": round((s[f"score_{a}"] == s[f"score_{b}"]).mean(), 3),
               "peer vs non-peer": round(((s[f"score_{a}"] >= 1) == (s[f"score_{b}"] >= 1)).mean(), 3)}
              for i, a in enumerate(names) for b in names[i + 1:]])

unanimous: 11 of 36; with a majority grade: 32


,pair,exact,peer vs non-peer
0,riya vs barbara,0.333,0.611
1,riya vs matthew,0.333,0.500
2,barbara vs matthew,0.833,0.889


### 3.5 Round 3: Barbara (old form) and Matthew (rubric v2) on the same 262 pairs

In [8]:
dev, m2 = cohorts.load("dev_293"), cohorts.load("matthew_v2_262")
both = dev.merge(m2, on=["target_id", "comp_id"], suffixes=("_barbara", "_matthew"))
print(f"{len(both)} shared pairs; exact {(both.grade_barbara == both.grade_matthew).mean():.0%}, "
      f"peer vs non-peer {(both.peer_barbara == both.peer_matthew).mean():.0%}")
display(pd.crosstab(both["grade_barbara"], both["grade_matthew"], rownames=["Barbara"], colnames=["Matthew v2"]))
q = ["same_product_matthew", "same_customers_matthew", "same_revenue_model_matthew", "same_chain_step_matthew"]
(m2[q] == "True").mean().round(2).rename("yes rate (Matthew)").to_frame().T

262 shared pairs; exact 51%, peer vs non-peer 65%


Matthew v2,0.0,1.0,2.0
Barbara,,,
0,36,48,1
1,42,76,38
2,0,0,21


,same_product_matthew,same_customers_matthew,same_revenue_model_matthew,same_chain_step_matthew
yes rate (Matthew),0.24,0.7,0.79,0.77


### 3.6 Barbara's 293 and the agent loop's grade (saved beside her labels)

In [9]:
pd.Series(json.loads(paths.src("06_labels", "barbara_293", "review", "summary.json").read_text()))

cohort              Barbara single reviewer; not consensus gold
pairs                                                       293
targets                                                      45
agent_scored                                                292
exact_agreement                                        0.839041
binary_agreement                                       0.873288
disagreements                                                47
dtype: object

### 3.7 Rubric v3: Matthew and Barbara on the same 311 pairs

Both answered all 323 pairs of the blind v3 sheet; the 12 anchors are left out of every number below. Run pinned
in `current.toml [rubric_v3] scores`; the tie-break is in 3.8. Kappa per question, with each reviewer's yes rate;
`unsure` counts pairs where either reviewer ticked that question.


In [10]:
V3 = paths.path(cur["rubric_v3"]["scores"])
agree = pd.read_csv(V3 / "agreement.csv")
agree.drop(columns="reviewers").round(3)


,item,pairs,agreement,kappa,yes_rate_matthew,yes_rate_barbara,unsure
0,overall_call,311,0.887,0.756,0.354,0.370,42.0
1,same_product,311,0.797,0.607,0.357,0.553,49.0
2,same_customers,311,0.791,0.574,0.550,0.598,46.0
3,same_revenue_model,311,0.955,0.861,0.801,0.794,30.0
4,same_chain_step,311,0.881,0.632,0.768,0.830,7.0
5,peer,311,0.785,0.560,0.550,0.605,NaN
6,direct,311,0.887,0.764,0.328,0.434,NaN
7,score,311,0.733,0.661,NaN,NaN,NaN


**Overall call against the derived score.** How often each reviewer put a pair in the comp set, by the score
their own four answers give.


In [11]:
bvd = pd.read_csv(V3 / "overall_vs_derived.csv").set_index("reviewer")
calls = pd.DataFrame({r: {f"call yes, score {k}": f'{bvd.at[r, f"score{k}_call_yes"]} of '
                          f'{bvd.at[r, f"score{k}_call_yes"] + bvd.at[r, f"score{k}_call_no"]}'
                      for k in (0, 1, 2)} for r in bvd.index}).T
display(calls)
bvd[["agree_peer", "kappa_peer", "agree_direct", "kappa_direct"]].round(3)


,"call yes, score 0","call yes, score 1","call yes, score 2"
matthew,3 of 140,7 of 69,100 of 102
barbara,0 of 123,11 of 53,104 of 135


,agree_peer,kappa_peer,agree_direct,kappa_direct
reviewer,,,,
matthew,0.785,0.581,0.961,0.914
barbara,0.765,0.555,0.865,0.720


**Derived scores side by side, and where the two split.** Rows Matthew, columns Barbara. Then, per question,
how many pairs one reviewer said yes to while the other said no.


In [12]:
m3, b3, a3 = cohorts.load("matthew_v3_311"), cohorts.load("barbara_v3_311"), cohorts.load("v3_agreed")
display(pd.crosstab(m3["grade"], b3["grade"], rownames=["Matthew v3"], colnames=["Barbara v3"], margins=True))
Q = ["overall_call", "same_product", "same_customers", "same_revenue_model", "same_chain_step"]
split = pd.DataFrame({q: {"Matthew yes, Barbara no": int((m3[f"{q}_matthew"] & ~b3[f"{q}_barbara"]).sum()),
                          "Matthew no, Barbara yes": int((~m3[f"{q}_matthew"] & b3[f"{q}_barbara"]).sum())}
                      for q in Q}).T
display(split)
print(f"agreed on all four questions: {a3.grade.notna().sum()} of {len(a3)} pairs "
      f"({a3.grade.value_counts().sort_index().to_dict()}); same derived score: {(m3.grade == b3.grade).sum()}")


Barbara v3,0.0,1.0,2.0,All
Matthew v3,,,,
0.0,98,23,19,140
1.0,25,29,15,69
2.0,0,1,101,102
All,123,53,135,311


,"Matthew yes, Barbara no","Matthew no, Barbara yes"
overall_call,15,20
same_product,1,62
same_customers,25,40
same_revenue_model,8,6
same_chain_step,9,28


agreed on all four questions: 180 of 311 pairs ({0.0: 71, 1.0: 8, 2.0: 101}); same derived score: 228


**Anchors.** Each reviewer's answer to the anchor's own question, next to the proposed ruling (`RUBRIC_V3.md`).
The reviewers had the guide and its rulings when they answered, so this checks reading, not independent agreement.


In [13]:
sc = pd.read_csv(V3 / "scores.csv", dtype={"target_id": str, "comp_id": str})
man = json.loads(paths.path(cur["rubric_v3"]["sheet"]).with_name("manifest.json").read_text(encoding="utf-8"))
yn = lambda v: "yes" if v else "no"
rows = []
for a in man["anchor_pairs"]:
    r = sc[(sc.target_id == a["target_id"]) & (sc.comp_id == a["comp_id"])].iloc[0]
    q = a["question"]
    rows.append({"pair": r.pair, "question": q, "kind": a["kind"], "proposed": a["proposed_answer"],
                 "Matthew": yn(r[f"{q}_matthew"]), "Barbara": yn(r[f"{q}_barbara"]),
                 "score M": int(r.score_matthew), "score B": int(r.score_barbara)})
anchors = pd.DataFrame(rows)
anchors["both match"] = (anchors.Matthew == anchors.proposed) & (anchors.Barbara == anchors.proposed)
anchors


,pair,question,kind,proposed,Matthew,Barbara,score M,score B,both match
0,P287,same_product,clear yes,yes,yes,yes,2,2,True
1,P254,same_product,clear no,no,no,no,0,0,True
2,P156,same_product,borderline,no,no,no,1,0,True
3,P318,same_customers,clear yes,yes,yes,no,1,0,False
4,P095,same_customers,clear no,no,no,no,0,0,True
5,P109,same_customers,borderline,no,no,no,0,0,True
6,P211,same_revenue_model,clear yes,yes,yes,yes,2,2,True
7,P181,same_revenue_model,clear no,no,no,no,0,0,True
8,P117,same_revenue_model,borderline,no,no,no,1,1,True
9,P018,same_chain_step,clear yes,yes,yes,yes,2,2,True


**What changed from the earlier labels**, same pairs: Barbara's old 0/1/2 grade (`dev_293`) against her v3
score, and Matthew's rubric v2 score (`matthew_v2_262`) against his v3 score.


In [14]:
k = ["target_id", "comp_id"]
ob = dev[k + ["grade"]].merge(b3[k + ["grade"]], on=k, suffixes=("_old", "_v3"))
display(pd.crosstab(ob.grade_old, ob.grade_v3, rownames=["Barbara old form"], colnames=["Barbara v3"],
                    margins=True))
om = m2[k + ["grade", "same_product_matthew", "same_customers_matthew"]].merge(
    m3[k + ["grade", "same_customers_matthew"]], on=k, suffixes=("_v2", "_v3"))
display(pd.crosstab(om.grade_v2, om.grade_v3, rownames=["Matthew v2"], colnames=["Matthew v3"], margins=True))
pc = om[(om.same_product_matthew == "False") & (om.same_customers_matthew_v2 == "True")]
print(f"Matthew's v2 'product no, customers yes' pairs: {len(pc)}; customers still yes in v3: "
      f"{int(pc.same_customers_matthew_v3.sum())}; v3 score {pc.grade_v3.value_counts().sort_index().to_dict()}")


Barbara v3,0.0,1.0,2.0,All
Barbara old form,,,,
0,74,9,4,87
1,49,43,81,173
2,0,0,21,21
All,123,52,106,281


Matthew v3,0.0,1.0,2.0,All
Matthew v2,,,,
0.0,60,13,3,76
1.0,65,44,8,117
2.0,0,4,53,57
All,125,61,64,250


Matthew's v2 'product no, customers yes' pairs: 116; customers still yes in v3: 51; v3 score {0.0: 65, 1.0: 43, 2.0: 8}


### 3.8 Rubric v3 tie-break: GPT-Terra, and the headline labels (`v3_final`)

Team ruling 2026-11-27 (`RUBRIC_V3.md`): agreement is per **outcome** (direct = score 2, peer = score 1 or 2, and
the overall call). Where Matthew and Barbara reach the same outcome it stands; where they split, the LLM judge's
outcome decides (2 of 3). The judge (`gpt-5.6-terra` through the Codex CLI, prompt `2026-11-27a`, `judge_v3.py`)
answered all 323 pairs blind, under the guide with the 11-27 rulings (question 1 on the comp's main business;
question 2 "the same buyers, in the same role"). Runs pinned in `current.toml [rubric_v3] judge` and `final`.

First: the judge against each reviewer, kappa per item (all 311 scored pairs).

In [15]:
J, F = paths.path(cur["rubric_v3"]["judge"]), paths.path(cur["rubric_v3"]["final"])
ja = pd.read_csv(J / "scores" / "agreement.csv", encoding="utf-8-sig")
ja.pivot_table(index="item", columns="reviewers", values="kappa").round(2)

reviewers,barbara vs terra,matthew vs barbara,matthew vs terra
item,,,
direct,0.70,0.76,0.84
overall_call,0.66,0.76,0.76
peer,0.50,0.56,0.65
same_chain_step,0.50,0.63,0.79
same_customers,0.50,0.57,0.65
same_product,0.53,0.61,0.81
same_revenue_model,0.88,0.86,0.89
score,0.60,0.66,0.74


**How far to trust it.** Where the two reviewers agree on an outcome, how often the judge gives the same one.
Then the anchors (it saw their rulings, so this only checks that it read the guide).

In [16]:
import rubric_v3
display(pd.read_csv(F / "judge_vs_agreed.csv").round(3))
js = rubric_v3.scores(rubric_v3._read(J / "filled_with_terra.csv"))
anc = js[js["is_anchor"].astype(bool)]
anc[["pair", *[f"{q}_terra" for q in rubric_v3.QUESTIONS], "score_terra", "score_matthew", "score_barbara"]]

,outcome,agreed_pairs,agreement,kappa,tie_broken,still_open
0,direct,276,0.946,0.879,35,0
1,peer,244,0.877,0.741,67,0
2,overall_call,276,0.906,0.800,35,0


,pair,same_product_terra,same_customers_terra,same_revenue_model_terra,same_chain_step_terra,score_terra,score_matthew,score_barbara
17,P018,True,True,True,True,2.0,2.0,2.0
94,P095,False,False,False,False,0.0,0.0,0.0
108,P109,False,False,True,True,0.0,0.0,0.0
116,P117,False,True,False,False,1.0,1.0,1.0
155,P156,False,True,True,True,1.0,1.0,0.0
168,P169,False,True,True,False,1.0,0.0,1.0
180,P181,False,False,False,True,0.0,0.0,0.0
210,P211,True,True,True,True,2.0,2.0,2.0
223,P224,True,True,False,False,1.0,1.0,1.0
253,P254,False,False,True,True,0.0,0.0,0.0


**The tie-breaks, and the partial-overlap answers.** Which reviewer the judge sided with on each split outcome; and on
the 62 pairs where Barbara said same product and Matthew did not (partial overlaps), the judge's product answer.

In [17]:
fin = pd.read_csv(F / "final.csv", encoding="utf-8-sig")
fin = fin[~fin["is_anchor"].astype(bool)]
s = js[~js["is_anchor"].astype(bool)].set_index("pair")
rows = []
for o, hit in (("direct", lambda v: v.eq(2)), ("peer", lambda v: v.ge(1)), ("overall_call", None)):
    t = fin[fin[f"{o}_source"].str.startswith("tiebreak")]
    m = (s.loc[t["pair"], "overall_call_matthew"].astype(bool) if hit is None else hit(s.loc[t["pair"], "score_matthew"])).to_numpy()
    rows.append({"outcome": o, "split pairs": len(t), "judge sided with Matthew": int((m == t[o].astype(bool).to_numpy()).sum()),
                 "judge sided with Barbara": int((m != t[o].astype(bool).to_numpy()).sum())})
display(pd.DataFrame(rows))
po = s[(s["same_product_barbara"] == True) & (s["same_product_matthew"] == False)]
pd.Series({"Barbara yes, Matthew no": len(po), "judge yes": int(po["same_product_terra"].sum())}, name="same product")

,outcome,split pairs,judge sided with Matthew,judge sided with Barbara
0,direct,35,29,6
1,peer,67,44,23
2,overall_call,35,25,10


Barbara yes, Matthew no    62
judge yes                   6
Name: same product, dtype: int64

**The headline labels** (`v3_final`): where each outcome came from, and the final grades.

In [18]:
src = pd.DataFrame({o: fin[f"{o}_source"].value_counts() for o in ("direct", "peer", "overall_call")}).fillna(0).astype(int)
display(src)
v = cohorts.load("v3_final")
pd.Series({"pairs": len(v), "targets": v["target_id"].nunique(), "direct (2)": int(v["grade"].eq(2).sum()),
           "adjacent (1)": int(v["grade"].eq(1).sum()), "not a peer (0)": int(v["grade"].eq(0).sum())})

,direct,peer,overall_call
agreed,276,244,276
tiebreak:terra,35,67,35


pairs             311
targets            67
direct (2)        106
adjacent (1)       78
not a peer (0)    127
dtype: int64

## 4. Checks

In [19]:
import contextlib, io, tempfile
import review_labels, round3_sheet, reviewer_gold_eval, rubric_v2
def same(a, b, **kw):
    x, y = pd.read_csv(a, dtype=str, **kw).fillna(""), pd.read_csv(b, dtype=str, **kw).fillna("")
    return x.equals(y)
R = paths.V1A_OUTPUTS / "06_labels"   # v2: the saved tables these rebuilds are checked against
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    with contextlib.redirect_stdout(io.StringIO()):
        review_labels.main(["--out", str(tmp / "review")])
        round3_sheet.main(["--candidates", str(paths.path(cur["retrieval"]["candidates"])), "--out", str(tmp / "r3")])
        reviewer_gold_eval.run(reviewer_gold_eval.LABELS, reviewer_gold_eval.POOL, tmp / "rge")
        rubric_v2.main(["scores", str(R / "round3" / "round3_sheet_v2.csv"), str(tmp / "v2.csv")])
    checks.add("Barbara's 293 table rebuilds from her workbook",
               all(same(tmp / "review" / f, R / "barbara_293" / "review" / f) for f in ["pairs.csv", "confusion.csv", "disagreements.csv"]))
    checks.add("round-3 sheet and pairs rebuild (seed 2026)", same(tmp / "r3" / "round3_sheet.csv", R / "round3" / "round3_sheet.csv")
               and same(tmp / "r3" / "round3_pairs.csv", R / "round3" / "round3_pairs.csv"))
    checks.add("Matthew's rubric-v2 scores re-derive from his answers",
               same(tmp / "v2.csv", R / "round3" / "round3_scores_v2.csv", encoding="utf-8-sig"))
    v1_rge = sorted((paths.V1_OUTPUTS / "golden_set" / "reviewer_evaluation").glob("*.csv"))
    checks.add("round-1 reviewer evaluation reproduces v1", len(v1_rge) == 9 and all(same(tmp / "rge" / f.name, f) for f in v1_rge),
               f"{len(v1_rge)} tables")
c = cohorts.table(include_locked=True)
checks.add("cohort sizes (293 / 36 / 262 / 311 / 311 / 311 / 311 / 24)",
           c["pairs"].tolist() == [293, 36, 262, 311, 311, 311, 311, 24])
try:
    cohorts.load("final_holdout_24"); locked = False
except cohorts.LockedCohort:
    locked = True
checks.add("final_holdout_24 refuses to load outside 10", locked)
old = pd.read_csv(R / "pair_ledger" / "20261118T205008Z" / "sources.csv")
new = pd.read_csv(ledger / "sources.csv")
m = old.merge(new, on="source", suffixes=("", "_new"))
cols = ["pairs", "judged", "grade_2", "grade_1", "grade_0", "sha256"]
checks.add("ledger: the 33 v1 sources unchanged since 11-18", len(m) == 33 and all((m[k] == m[k + "_new"]).all() for k in cols),
           f"{len(new)} sources now")
import rubric_v3
V3 = paths.path(cur["rubric_v3"]["scores"])
filled = rubric_v3._read(paths.path(cur["rubric_v3"]["filled"]))
blank = rubric_v3._read(paths.path(cur["rubric_v3"]["sheet"]))
desc = [k for k in blank.columns if k in filled.columns and not any(k.endswith("_" + r) for r in rubric_v3.REVIEWERS)]
checks.add("rubric v3: filled sheet has the blank sheet's pairs and descriptions, unchanged",
           filled["pair"].tolist() == blank["pair"].tolist() and all(filled[k].equals(blank[k]) for k in desc),
           f"{len(desc)} columns")
checks.add("rubric v3: no blank or invalid answers (rubric_v3.py check)", len(rubric_v3.check_sheet(filled)) == 0)
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    with contextlib.redirect_stdout(io.StringIO()):
        rubric_v3.main(["scores", str(paths.path(cur["rubric_v3"]["filled"])), str(tmp)])
        rubric_v3.main(["consensus", str(paths.path(cur["rubric_v3"]["filled"])), str(tmp / "consensus.csv")])
    checks.add("rubric v3: the pinned scores run re-derives from the filled sheet",
               all(same(tmp / f, V3 / f, encoding="utf-8-sig") for f in ["scores.csv", "consensus.csv"])
               and all(same(tmp / f, V3 / f) for f in ["agreement.csv", "overall_vs_derived.csv"]))
J, F = paths.path(cur["rubric_v3"]["judge"]), paths.path(cur["rubric_v3"]["final"])
jf = rubric_v3._read(J / "filled_with_terra.csv")
checks.add("rubric v3 judge: answered all 323 pairs, no invalid answers",
           jf["same_product_terra"].astype(str).str.strip().ne("").sum() == 323 and len(rubric_v3.check_sheet(jf)) == 0)
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    with contextlib.redirect_stdout(io.StringIO()):
        rubric_v3.main(["final", str(J / "filled_with_terra.csv"), str(tmp), "--judge", "terra"])
    checks.add("rubric v3: the pinned final labels re-derive from the judge run",
               same(tmp / "final.csv", F / "final.csv", encoding="utf-8-sig") and same(tmp / "judge_vs_agreed.csv", F / "judge_vs_agreed.csv"))
fv = cohorts.load("v3_final")
checks.add("v3_final: 311 pairs, none left open", len(fv) == 311 and fv["grade"].notna().all())
checks.show()


13/13 checks pass


,check,result,detail
0,Barbara's 293 table rebuilds from her workbook,pass,
1,round-3 sheet and pairs rebuild (seed 2026),pass,
2,Matthew's rubric-v2 scores re-derive from his answers,pass,
3,round-1 reviewer evaluation reproduces v1,pass,9 tables
4,cohort sizes (293 / 36 / 262 / 311 / 311 / 311 / 311 / 24),pass,
5,final_holdout_24 refuses to load outside 10,pass,
6,ledger: the 33 v1 sources unchanged since 11-18,pass,34 sources now
7,"rubric v3: filled sheet has the blank sheet's pairs and descriptions, unchanged",pass,17 columns
8,rubric v3: no blank or invalid answers (rubric_v3.py check),pass,
9,rubric v3: the pinned scores run re-derives from the filled sheet,pass,


## 5. History (v1)

- **Round 1 (10-27/28):** Michael's sheet, then Riya, Barbara and Matthew on 36 PCTY/WEN pairs; 11 unanimous.
- **Round 2 (11-04):** a 67-pair sheet for CSL, AFG, AZZ, BBCP, BUKS + 2 WEN, three neutral reviewer slots, the six
  reasons and the multi-business rule (`GOLDEN_SET_ROUND2.md`). Never filled; superseded by the Sonnet-list rounds.
- **Rounds 2-3 on the Sonnet lists (11-05/06):** 31 + 262 pairs; Barbara graded all 293 (committed 11-10).
- **Rubric v2 (11-13, Matthew and Riya):** four yes/no questions, score derived (`RUBRIC_V2.md`); Matthew answered the
  262 round-3 pairs. Question 2 (same customers) decides most disagreements with Barbara's old form.
- **Barbara's direct peers (11-17):** 90 pairs she picked; the 24 with comps in the universe are the locked final
  holdout; the 66 with comps outside the universe are left out of v1a.
- `integration_sources.py` and Barbara's published-list scripts stay in v1 (they read files outside the repo); their
  outputs are seeded here as frozen sources (`barbara_293/sonnet_label.xlsx`, `barbara_published/`).